In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
pd.set_option('display.max_rows', None)

In [2]:
train = pd.read_parquet('data/train.parquet')
test = pd.read_parquet('data/test.parquet')
val = pd.read_parquet('data/val.parquet')

In [3]:
x_cols = ['Avg source IP count', 'Detect count_y', 'Victim IP_y', 'Port number_y', 
          'Packet speed_y', 'Data speed_y', 'Avg packet len_y', 'Source IP count', 
          'Packet speed_y_normalized', 'Data speed_y_normalized', 'time_of_day',
          'Avg packet len_y_normalized', 'total_seconds', 'weekday_number', 
          'IsWeekend', 'Start Hour', 'Sin_Hour', 'Cos_Hour', 'DayOfYear', 
          'Sin_DayOfYear', 'Cos_DayOfYear', 'Mean_DataSpeed', 'Std_DataSpeed', 
          'Min_DataSpeed', 'Max_DataSpeed', 'Mean_DetectCount', 'Std_DetectCount', 
          'Min_DetectCount', 'Max_DetectCount', 'VictimIP_Count', 'PortNumber_Count', 
          'AvgPacketLen_Mean', 'AvgPacketLen_Std', 'DataSpeed_PacketSpeed', 
          'PortFrequency', 'Std_DataSpeed_Replaced', 'Std_DetectCount_Replaced', 
          'AvgPacketLen_Std_Replaced', 'packet_Total', 'PacketSpeed_Per_Second', 
          'DataSpeed_Per_TotalSeconds', 'AvgPacketLen_Per_TotalSeconds', 
          'PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5', 'Is_HTTP', 'Is_HTTPS', 
          'Is_FTP_Control', 'Is_FTP_Data', 'Is_SSH', 'Is_Telnet', 'Is_SMTP', 
          'Is_DNS', 'Is_POP3', 'Is_IMAP', 'Is_DHCP', 'Is_SNMP', 'Is_LDAP', 
          'Is_LDAPS', 'Is_SMB_CIFS', 'Is_RDP', 'Is_SIP', 'Is_TFTP', 'Is_MySQL', 
          'Is_PostgreSQL', 'Is_Oracle', 'Is_HTTP_Alt_8080', 'Is_HTTP_Alt_8081', 
          'Is_HTTP_Alt_80', 'Is_HTTPS_Alt_8443', 'Is_Syslog', 'Is_VNC', 'Is_IRC', 
          'Is_NTP', 'Is_Kerberos', 'Is_LDAP_Alt', 'Is_LDAPS_Alt', 'Is_RADIUS', 
          'Is_PPTP', 'Is_RTSP', 'Is_X11', 'Is_SNMP_Trap', 'Is_BGP', 'Is_IMAPS_Alt', 
          'Is_POP3S_Alt', 'Is_Telnet_SSL', 'Is_NNTP', 'Is_NNTPS', 'Is_LDAP_TLS', 
          'Is_AFS', 'Is_NFS', 'Is_SOCKS', 'Is_RSYNC', 'Is_CUPS', 'Is_TFTP_Alt', 
          'Is_Modbus', 'Is_CoAP', 'Is_MQTT', 'Is_AMQP', 'Is_Redis', 'Is_Memcached', 
          'Is_Elasticsearch', 'Is_Zookeeper', 'Is_Cassandra', 'Is_Docker', 
          'Is_Kubernetes', 'Is_SMB_Direct', 'Is_iSCSI', 'Is_AFP', 'Is_DHCPv6', 
          'Is_RIPng', 'Is_OSPF', 'Is_PPPoE', 'Is_L2TP', 'Is_GRE', 'Is_ESP', 'Is_AH']

In [9]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score


def objective(trial):
    # Define the hyperparameters to optimize and their search spaces
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    max_depth = trial.suggest_int('max_depth', 2, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)

    # Create the Random Forest Classifier with the suggested hyperparameters
    rfc = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42, 
        verbose=0,
        )

    # Train the model
    rfc.fit(train[x_cols], train['Type'])

    # Make predictions on the test set
    y_pred = rfc.predict(test[x_cols])

    # Calculate the evaluation metric (accuracy in this case)
    accuracy = f1_score(test['Type'], y_pred, average='macro')
    return accuracy

if __name__ == "__main__":
    # Create a study object and specify the optimization direction
    study = optuna.create_study(direction='maximize')

    # Run the optimization process
    study.optimize(objective, n_trials=20, show_progress_bar=True)  # Number of trials to run

    # Print the best hyperparameters found
    print("Best hyperparameters:", study.best_params)

    # Get the best score achieved
    print("Best accuracy:", study.best_value)

    # Create a model with the best hyperparameters
    best_rfc = RandomForestClassifier(**study.best_params, random_state=42)

    # Train the best model on the full training data
    best_rfc.fit(train[x_cols], train['Type'])

    # Evaluate the best model on the test setí
    y_pred_best = best_rfc.predict(test[x_cols])
    best_accuracy = f1_score(test['Type'], y_pred_best, average='macro')
    print(f"Accuracy of the best model on the test set: {best_accuracy:.4f}")

    # You can also visualize the optimization history and parameter importance
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()

[I 2025-05-11 12:55:22,997] A new study created in memory with name: no-name-b4b6f5b8-fcfb-4ea1-98e6-2e3497616960


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2025-05-11 13:00:34,458] Trial 0 finished with value: 0.6063325699573671 and parameters: {'n_estimators': 51, 'max_depth': 36, 'min_samples_split': 5, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.6063325699573671.
[I 2025-05-11 13:41:13,344] Trial 1 finished with value: 0.5797656241176748 and parameters: {'n_estimators': 427, 'max_depth': 21, 'min_samples_split': 10, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.6063325699573671.
[I 2025-05-11 14:26:22,716] Trial 2 finished with value: 0.6151665397645224 and parameters: {'n_estimators': 437, 'max_depth': 37, 'min_samples_split': 10, 'min_samples_leaf': 5}. Best is trial 2 with value: 0.6151665397645224.
[I 2025-05-11 14:36:14,990] Trial 3 finished with value: 0.5975163406415449 and parameters: {'n_estimators': 97, 'max_depth': 36, 'min_samples_split': 10, 'min_samples_leaf': 4}. Best is trial 2 with value: 0.6151665397645224.
[I 2025-05-11 14:46:59,958] Trial 4 finished with value: 0.4434942262873533 and parameters:

KeyboardInterrupt: 

In [8]:
import optuna
import lightgbm as lgb
from sklearn.metrics import f1_score

def objective(trial):
    # Define the hyperparameters to optimize and their search spaces for LightGBM
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    num_leaves = trial.suggest_int('num_leaves', 31, 100)
    max_depth = trial.suggest_int('max_depth', -1, 10)  # -1 means no limit
    min_child_samples = trial.suggest_int('min_child_samples', 20, 50)
    subsample = trial.suggest_float('subsample', 0.7, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0)

    # Create the LightGBM Classifier with the suggested hyperparameters
    lgbm = lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=42,
        verbose=-1,
    )

    # Train the model
    lgbm.fit(train[x_cols], train['Type'])

    # Make predictions on the test set
    y_pred = lgbm.predict(test[x_cols])

    # Calculate the evaluation metric (accuracy in this case)
    accuracy = f1_score(test['Type'], y_pred, average='macro')
    return accuracy

if __name__ == "__main__":
    # Create a study object and specify the optimization direction
    study = optuna.create_study(direction='maximize')

    # Run the optimization process
    study.optimize(objective, n_trials=30, show_progress_bar=True)  # Number of trials to run

    # Print the best hyperparameters found
    print("Best hyperparameters:", study.best_params)

    # Get the best score achieved
    print("Best accuracy:", study.best_value)

    # Create a model with the best hyperparameters
    best_lgbm = lgb.LGBMClassifier(**study.best_params, random_state=42)

    # Train the best model on the full training data
    best_lgbm.fit(train[x_cols], train['Type'])

    # Evaluate the best model on the test set
    y_pred_best = best_lgbm.predict(test[x_cols])
    best_accuracy = f1_score(test['Type'], y_pred_best, average='macro')
    print(f"Accuracy of the best LightGBM model on the test set: {best_accuracy:.4f}")

    # You can also visualize the optimization history and parameter importance
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()

[I 2025-05-11 12:21:28,926] A new study created in memory with name: no-name-e3e016fe-ebd7-48ef-85d0-e7460d2da5ba


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2025-05-11 12:22:06,362] Trial 0 finished with value: 0.46684995951681335 and parameters: {'n_estimators': 233, 'learning_rate': 0.2365610130041753, 'num_leaves': 38, 'max_depth': 8, 'min_child_samples': 29, 'subsample': 0.8626325235150197, 'colsample_bytree': 0.7736082534188424}. Best is trial 0 with value: 0.46684995951681335.
[I 2025-05-11 12:23:28,463] Trial 1 finished with value: 0.5002133471865974 and parameters: {'n_estimators': 448, 'learning_rate': 0.2665034416286709, 'num_leaves': 71, 'max_depth': 8, 'min_child_samples': 21, 'subsample': 0.7727696879580079, 'colsample_bytree': 0.7555858441460386}. Best is trial 1 with value: 0.5002133471865974.
[I 2025-05-11 12:23:41,407] Trial 2 finished with value: 0.5371736795256862 and parameters: {'n_estimators': 132, 'learning_rate': 0.15606935164402186, 'num_leaves': 94, 'max_depth': 1, 'min_child_samples': 27, 'subsample': 0.7526844461142074, 'colsample_bytree': 0.7168373512531228}. Best is trial 2 with value: 0.5371736795256862.
[

In [ ]:
"""Best hyperparameters: {'n_estimators': 341, 'learning_rate': 0.05719243532859288, 'num_leaves': 47, 'max_depth': 5, 
'min_child_samples': 42, 'subsample': 0.8896113748808988, 'colsample_bytree': 0.9577482057904019}
Best f1: 0.6048903438655178
"""

In [ ]:
import optuna
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

def objective(trial):
    # Create the XGBM Classifier with the suggested hyperparameters
    xgb = XGBClassifier(
        objective='multi:softmax',
        num_class=3,
        booster=trial.suggest_categorical('booster', ['gbtree', 'gblinear', 'dart']),
        alpha=trial.suggest_float('alpha', 1e-8, 1.0),
        subsample=trial.suggest_float('subsample', 0.2, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.2, 1.0),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        eta=trial.suggest_float('eta', 0.005, 0.3),
        gamma=trial.suggest_float('gamma', 1e-8, 1.0),
        grow_policy=trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        eval_metric='merror',  # Multi-class error for evaluation
        random_state=42,
    )

    # Train the model
    xgb.fit(train[x_cols], train['Type'])

    # Make predictions on the test set
    y_pred = xgb.predict(test[x_cols])

    # Calculate the evaluation metric (accuracy in this case)
    accuracy = f1_score(test['Type'], y_pred, average='macro')
    return accuracy

if __name__ == "__main__":
    # Create a study object and specify the optimization direction
    study = optuna.create_study(direction='maximize')

    # Run the optimization process
    study.optimize(objective, n_trials=50, show_progress_bar=True)  # Number of trials to run

    # Print the best hyperparameters found
    print("Best hyperparameters:", study.best_params)

    # Get the best score achieved
    print("Best accuracy:", study.best_value)

    # Create a model with the best hyperparameters
    best_xgb = xgb.XGBClassifier(**study.best_params, random_state=42)

    # Train the best model on the full training data
    best_lgbm.fit(train[x_cols], train['Type'])

    # Evaluate the best model on the test set
    y_pred_best = best_lgbm.predict(test[x_cols])
    best_accuracy = f1_score(test['Type'], y_pred_best, average='macro')
    print(f"Accuracy of the best LightGBM model on the test set: {best_accuracy:.4f}")

    # You can also visualize the optimization history and parameter importance
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()

[I 2025-05-11 03:20:58,532] A new study created in memory with name: no-name-86974e30-7ba5-4fcd-b30c-8361720bfb09


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-11 03:21:35,452] Trial 0 finished with value: 0.5812686328189911 and parameters: {'booster': 'gbtree', 'alpha': 0.18079280970256723, 'subsample': 0.6588565259866888, 'colsample_bytree': 0.6848678073647998, 'max_depth': 5, 'eta': 0.171241187438225, 'gamma': 0.9842572030054573, 'grow_policy': 'depthwise', 'min_child_weight': 9}. Best is trial 0 with value: 0.5812686328189911.
[I 2025-05-11 04:51:05,624] Trial 1 finished with value: 0.5657683742494699 and parameters: {'booster': 'dart', 'alpha': 0.057300849199275666, 'subsample': 0.42354551373064303, 'colsample_bytree': 0.5409354437009269, 'max_depth': 4, 'eta': 0.18774184012216844, 'gamma': 0.8110058152397175, 'grow_policy': 'depthwise', 'min_child_weight': 1}. Best is trial 0 with value: 0.5812686328189911.


c:\Users\Admin\anaconda3\envs\DS\Lib\site-packages\xgboost\core.py:158: UserWarning: [04:51:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "colsample_bytree", "gamma", "grow_policy", "max_depth", "min_child_weight", "subsample" } are not used.

  warnings.warn(smsg, UserWarning)


[I 2025-05-11 04:52:53,571] Trial 2 finished with value: 0.45266335360965265 and parameters: {'booster': 'gblinear', 'alpha': 0.15853103190987852, 'subsample': 0.45145876206554303, 'colsample_bytree': 0.48724279176976504, 'max_depth': 5, 'eta': 0.24226594397802564, 'gamma': 0.2659602951430409, 'grow_policy': 'depthwise', 'min_child_weight': 1}. Best is trial 0 with value: 0.5812686328189911.


...

[I 2025-05-11 09:34:42,210] Trial 19 finished with value: 0.452682187562242 and parameters: {'booster': 'gblinear', 'alpha': 0.7067245016223458, 'subsample': 0.511040739965053, 'colsample_bytree': 0.5501435063042703, 'max_depth': 7, 'eta': 0.0862427435798321, 'gamma': 0.8890617227488672, 'grow_policy': 'depthwise', 'min_child_weight': 8}. Best is trial 12 with value: 0.5888544949699649.
[I 2025-05-11 09:35:24,162] Trial 20 finished with value: 0.596447126654453 and parameters: {'booster': 'gbtree', 'alpha': 0.48959348267976266, 'subsample': 0.39187723779794587, 'colsample_bytree': 0.47468834851794295, 'max_depth': 9, 'eta': 0.03248507158076952, 'gamma': 0.7135042006424785, 'grow_policy': 'depthwise', 'min_child_weight': 3}. Best is trial 20 with value: 0.596447126654453.
[I 2025-05-11 09:36:06,094] Trial 21 finished with value: 0.5908390826744793 and parameters: {'booster': 'gbtree', 'alpha': 0.4916999777810437, 'subsample': 0.3827985872830677, 'colsample_bytree': 0.44572418365044775, 

c:\Users\Admin\anaconda3\envs\DS\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:09:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "colsample_bytree", "gamma", "grow_policy", "max_depth", "min_child_weight", "subsample" } are not used.

  warnings.warn(smsg, UserWarning)


[I 2025-05-11 11:11:31,204] Trial 29 finished with value: 0.45364465962052486 and parameters: {'booster': 'gblinear', 'alpha': 0.8643285522335586, 'subsample': 0.6984688460062742, 'colsample_bytree': 0.4047956631186724, 'max_depth': 7, 'eta': 0.1369169298466193, 'gamma': 0.10870293724641228, 'grow_policy': 'lossguide', 'min_child_weight': 2}. Best is trial 26 with value: 0.5978143690983727.
[I 2025-05-11 11:12:06,181] Trial 30 finished with value: 0.5643890299003104 and parameters: {'booster': 'gbtree', 'alpha': 0.9970282062156645, 'subsample': 0.6373580306453941, 'colsample_bytree': 0.7659350230239166, 'max_depth': 6, 'eta': 0.030112349009069034, 'gamma': 0.4995051706087036, 'grow_policy': 'lossguide', 'min_child_weight': 4}. Best is trial 26 with value: 0.5978143690983727.
[I 2025-05-11 11:12:50,449] Trial 31 finished with value: 0.583650959836659 and parameters: {'booster': 'gbtree', 'alpha': 0.653141123841095, 'subsample': 0.47710875489423493, 'colsample_bytree': 0.4330848605787507

c:\Users\Admin\anaconda3\envs\DS\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:16:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "colsample_bytree", "gamma", "grow_policy", "max_depth", "min_child_weight", "subsample" } are not used.

  warnings.warn(smsg, UserWarning)


[I 2025-05-11 11:17:48,705] Trial 36 finished with value: 0.4514307352792057 and parameters: {'booster': 'gblinear', 'alpha': 0.9288015495799496, 'subsample': 0.42998380250963314, 'colsample_bytree': 0.4834058045447235, 'max_depth': 9, 'eta': 0.04822154106734155, 'gamma': 0.29562506007055456, 'grow_policy': 'lossguide', 'min_child_weight': 1}. Best is trial 26 with value: 0.5978143690983727.


In [ ]:
"""{'booster': 'gbtree', 'alpha': 0.8929914624140356, 'subsample': 0.5245590585473099, 'colsample_bytree': 0.41745486572446977, 'max_depth': 10, 
'eta': 0.03343544751435021, 'gamma': 0.36103800824329335, 'grow_policy': 'lossguide', 'min_child_weight': 2}. Best is trial 26 with value: 0.5978143690983727."""

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
best_xgb = XGBClassifier(
    objective='multi:softmax',
        num_class=3,
        booster=('gbtree'),
        alpha=0.8929914624140356,
        subsample=0.5245590585473099,
        colsample_bytree=0.41745486572446977,
        max_depth=10,
        eta=0.03343544751435021,
        gamma=0.36103800824329335,
        grow_policy='lossguide',
        min_child_weight=2,
        eval_metric='merror',  # Multi-class error for evaluation
        random_state=42,)

# Train the best model on the full training data
best_xgb.fit(train[x_cols], train['Type'])

# Evaluate the best model on the test set
y_pred_best = best_xgb.predict(test[x_cols])
best_accuracy = f1_score(test['Type'], y_pred_best, average='macro')
print(f"Accuracy of the best LightGBM model on the test set: {best_accuracy:.4f}")

Accuracy of the best LightGBM model on the test set: 0.5978


NameError: name 'optuna' is not defined